# HMapp algorithme

This notebook attempts to reproduce the findings of the 2026 paper, more specifically the pipeline used to find these results. A few modifications will be made:
 - Use a already selected corpus of texts, thus skipping entirely the seed search as performed by the HMapp
 - Limited result testing: for now, lack of viable dataset

In [27]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import bigrams, trigrams

## Load the dataset

In [28]:
# load the data
df = pd.read_csv("./data/Unmasking Antisemitism SRI Data Set - Reporting Layer.csv")

In [29]:
df.describe()



,Time,Likes,Shares,Hateful,Violent,Offensive,Negative,Homophobic,Transphobic,Racist,Misogynistic,Xenophobic
count,0.0,295.000000,123.000000,659.000000,659.000000,659.000000,659.000000,660.000000,660.000000,660.000000,660.000000,660.000000
mean,NaN,11.301695,3.186992,0.335102,0.342416,0.491892,0.817020,0.075758,0.063636,0.227273,0.024242,0.096970
std,NaN,54.013797,16.545689,0.122211,0.312372,0.074944,0.265517,0.990250,0.834869,2.932190,0.329648,1.262145
min,NaN,0.000000,0.000000,0.159000,0.022000,0.397000,0.027000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.000000,0.250000,0.108000,0.458500,0.839500,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,1.000000,0.000000,0.309000,0.199000,0.479000,0.935000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,4.000000,1.000000,0.380500,0.515500,0.506000,0.967000,0.000000,0.000000,0.000000,0.000000,0.000000
max,NaN,584.000000,174.000000,0.859000,0.971000,0.991000,0.991000,25.000000,21.000000,75.000000,8.000000,32.000000


In [30]:
df.shape

(667, 29)

In [31]:
df.head()

,Term or Phrase,Date,Time,User,User Affiliation,Media Platform,Link to Post,Post Text,Screenshot,Image,...,Homophobic,Transphobic,Racist,Misogynistic,Xenophobic,Trope Classification,Primary Coder,Coder 2,Coder 3,Notes
0,Cabal,12/06/22,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@GlyMor/10946663297134...,"Exactly!!!! Bring in the mighty Marines, round...",Cabal_01,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,NaN
1,Cabal,1/19/2023,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@17QRanches/1097135105...,WINNING üí•WINNING üí•WINNING üí•CLEANING U...,Cabal_02,https://static-assets-1.truthsocial.com/tmtg:p...,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,Target is RHINOS
2,Cabal,12/27/2022,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@1776WeThePeople1776/1...,Mr President this Cabal must all be destroyed!...,Cabal_03,https://static-assets-1.truthsocial.com/tmtg:p...,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,NaN
3,Cabal,12/06/22,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@blakecrosby/109466136...,ITS VAX GENOCIDE A COLOR REVOLUTION AND THE CA...,Cabal_04,NaN,...,0.0,0.0,0.0,0.0,0.0,"1,8",coder 1,coder 2,coder 3,Jews as servants of Satan
4,Cabal,12/14/2022,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@QuirkyOtter/109513112...,"Yes, Because CHILD PORNOGRAPHY Global Cabal! R...",Cabal_05,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,NaN


## Candidate extraction

Now that are corpus is available to us, we can extract candidate terms as described

### Text pre-processing
This step includes removing hyperlinks, stop words, setting everything in lowercase, and constructing unigrams, bigrams and trigrams.

In [32]:
# copy our df
data = df.copy()
### cleaning ####
# remove hyperlinks
data['Post Text'] = data['Post Text'].str.replace(r'http\S+|www\S+|https\S+', '', regex=True, case=False)

# lowercase entire text
data['Post Text'] = data['Post Text'].str.lower()

# remove stopwords
# first, load stop words
nltk.download("stopwords")
nltk.download("punkt_tab") # have to download this resource as well for tokenization; not sure why

stop_words = set(stopwords.words('english'))

# remove stop words from the text
# use a lambda function for ease
# Preserve missing values when processing text.
def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    tokens = word_tokenize(text)
    return ' '.join([word for word in tokens if word not in stop_words])

data["Post Text Without Stopwords"] = data["Post Text"].apply(remove_stopwords)

print(f"Before stop word removal: \n{df['Post Text'].head()}")
print(f"After stop word removal: \n{data['Post Text Without Stopwords'].head()}")

[nltk_data] Downloading package stopwords to /home/armand/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/armand/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Before stop word removal: 
0    Exactly!!!! Bring in the mighty Marines, round...
1    WINNING üí•WINNING üí•WINNING üí•CLEANING U...
2    Mr President this Cabal must all be destroyed!...
3    ITS VAX GENOCIDE A COLOR REVOLUTION AND THE CA...
4    Yes, Because CHILD PORNOGRAPHY Global Cabal! R...
Name: Post Text, dtype: str
After stop word removal: 
0    exactly ! ! ! ! bring mighty marines , round p...
1    winning üí•winning üí•winning üí•cleaning r...
2    mr president cabal must destroyed ! ! period !...
3    vax genocide color revolution cabal laughing w...
4    yes , child pornography global cabal ! right f...
Name: Post Text Without Stopwords, dtype: str


In [33]:
# construct unigram, bigram and trigram features
# def: a n-gram is a sequence of n adjacent words

# thus, we have to do word-token tokenisation first, then we can generate n-grams

test = data['Post Text Without Stopwords'].iloc[0]
tokens = word_tokenize(test)
unigrams_list = tokens
bigram_list = list(bigrams(tokens))
trigram_list = list(trigrams(tokens))

data["Unigrams"] = data["Post Text Without Stopwords"].apply(
    lambda text: word_tokenize(text) if isinstance(text, str) else []
)
data["Bigrams"] = data["Unigrams"].apply(
    lambda tokens: list(bigrams(tokens))
)
data["Trigrams"] = data["Unigrams"].apply(
    lambda tokens: list(trigrams(tokens))
)


print(f"Unigrams: \n{data['Unigrams'].head()}")
print(f"Bigrams: \n{data['Bigrams'].head()}")
print(f"Trigrams: \n{data['Trigrams'].head()}")

Unigrams: 
0    [exactly, !, !, !, !, bring, mighty, marines, ...
1    [winning, üí•winning, üí•winning, üí•cleani...
2    [mr, president, cabal, must, destroyed, !, !, ...
3    [vax, genocide, color, revolution, cabal, laug...
4    [yes, ,, child, pornography, global, cabal, !,...
Name: Unigrams, dtype: object
Bigrams: 
0    [(exactly, !), (!, !), (!, !), (!, !), (!, bri...
1    [(winning, üí•winning), (üí•winning, üí•win...
2    [(mr, president), (president, cabal), (cabal, ...
3    [(vax, genocide), (genocide, color), (color, r...
4    [(yes, ,), (,, child), (child, pornography), (...
Name: Bigrams, dtype: object
Trigrams: 
0    [(exactly, !, !), (!, !, !), (!, !, !), (!, !,...
1    [(winning, üí•winning, üí•winning), (üí•win...
2    [(mr, president, cabal), (president, cabal, mu...
3    [(vax, genocide, color), (genocide, color, rev...
4    [(yes, ,, child), (,, child, pornography), (ch...
Name: Trigrams, dtype: object


In [34]:
# Tag the existing unigram, bigram, and trigram candidates.
nltk.download("averaged_perceptron_tagger_eng")

# Unigrams are strings; bigrams and trigrams are tuples of words.
data["Unigrams POS"] = data["Unigrams"].apply(
    lambda candidates: [nltk.pos_tag([word]) for word in candidates]
)
data["Bigrams POS"] = data["Bigrams"].apply(
    lambda candidates: [nltk.pos_tag(list(gram)) for gram in candidates]
)
data["Trigrams POS"] = data["Trigrams"].apply(
    lambda candidates: [nltk.pos_tag(list(gram)) for gram in candidates]
)

# Each tagged candidate contains (word, POS tag) pairs.
data[["Unigrams POS", "Bigrams POS", "Trigrams POS"]].head()


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/armand/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


,Unigrams POS,Bigrams POS,Trigrams POS
0,"[[(exactly, RB)], [(!, .)], [(!, .)], [(!, .)]...","[[(exactly, RB), (!, .)], [(!, .), (!, .)], [(...","[[(exactly, RB), (!, .), (!, .)], [(!, .), (!,..."
1,"[[(winning, VBG)], [(üí•winning, VBG)], [(üí...","[[(winning, VBG), (üí•winning, VBG)], [(üí•w...","[[(winning, VBG), (üí•winning, VBG), (üí•win..."
2,"[[(mr, NN)], [(president, NN)], [(cabal, NN)],...","[[(mr, NN), (president, NN)], [(president, NN)...","[[(mr, NN), (president, NN), (cabal, NN)], [(p..."
3,"[[(vax, NN)], [(genocide, NN)], [(color, NN)],...","[[(vax, NNS), (genocide, VBP)], [(genocide, NN...","[[(vax, NN), (genocide, NN), (color, NN)], [(g..."
4,"[[(yes, NNS)], [(,, ,)], [(child, NN)], [(porn...","[[(yes, NNS), (,, ,)], [(,, ,), (child, NN)], ...","[[(yes, UH), (,, ,), (child, NN)], [(,, ,), (c..."


In [35]:
# Provisional POS rules: these are not verified HMApp rules.
# Keep single nouns/adjectives, and noun/adjective phrases ending in a noun.
# Penn Treebank tags: NN* = nouns; JJ* = adjectives.
def keep_candidate(tagged_gram):
    tags = [tag for word, tag in tagged_gram]
    if not tags:
        return False
    if len(tags) == 1:
        return tags[0].startswith(("NN", "JJ"))
    return (
        all(tag.startswith(("NN", "JJ")) for tag in tags)
        and tags[-1].startswith("NN")
    )

# Preserve original candidates and store survivors separately.
for column in ("Unigrams", "Bigrams", "Trigrams"):
    def filter_candidates(tagged_candidates):
        kept = []
        for gram in tagged_candidates:
            if keep_candidate(gram):
                words = tuple(word for word, tag in gram)
                kept.append(words[0] if len(words) == 1 else words)
        return kept

    data[f"{column} Filtered"] = data[f"{column} POS"].apply(filter_candidates)
    before = data[column].apply(len).sum()
    after = data[f"{column} Filtered"].apply(len).sum()
    print(f"{column}: {before} occurrences before filtering, {after} after")

data[["Unigrams Filtered", "Bigrams Filtered", "Trigrams Filtered"]].head()


Unigrams: 50221 occurrences before filtering, 32371 after
Bigrams: 49569 occurrences before filtering, 17199 after
Trigrams: 48919 occurrences before filtering, 9555 after


,Unigrams Filtered,Bigrams Filtered,Trigrams Filtered
0,"[bring, mighty, marines, round, purge, hell, c...","[(bring, mighty), (mighty, marines), (round, p...","[(round, purge, hell), (purge, hell, corrupt)]"
1,"[rats, üêä, @, 17qranches, q, truth, truths, ...","[(üêä, @)]",[]
2,"[mr, president, cabal, destroyed, period, üá∫...","[(mr, president), (president, cabal)]","[(mr, president, cabal)]"
3,"[vax, genocide, color, revolution, cabal, way,...","[(genocide, color), (color, revolution), (revo...","[(vax, genocide, color), (genocide, color, rev..."
4,"[yes, child, pornography, global, cabal, right...","[(child, pornography), (global, cabal), (right...","[(pornography, global, cabal), (california, go..."


## Candidate Discovery
Now that the corpus has been pre-processed, we can now apply our candidate discovery methods
1) TF-IDF
2) Build contextual embeddings for anchor occurrences
3) Build contextual embeddings for candidate occurrences
4) Compare each candidate occurrence against the anchor database
5) Aggregate across all occurrences of the candidate
6) Final filtering/ranking

In [ ]:
### TF-IDF ###
# For every term t, compute its TF-IDF in every post, then keep its maximum TF-IDF over all posts
# then compute hen compute the mean of all these maxima and keep all terms where their TF-IDF is greater than the mean

